# Hyperparameter Tuning

**Objective:** Find optimal hyperparameters for Random Forest, XGBoost, and LightGBM using GridSearchCV with TimeSeriesSplit cross-validation.

**Approach:**
- Use 20% sample of training data for faster search
- TimeSeriesSplit with 3 folds (respects temporal order)
- Score: R² (consistent with evaluation)

**Input:** `data/processed/final_featured_dataset.csv`  
**Output:** `results/best_hyperparameters.json`, `results/grid_search_results.csv`

## Import Required Libraries


In [13]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
import time as timer
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

print("Libraries loaded!")

Libraries loaded!


## 1. Load Data & Prepare Split

### 1.1 Load & Concat Parts

In [14]:
DATA_DIR = Path("../data/processed")

files = sorted(DATA_DIR.glob("final_featured_dataset_part_*.csv"))
df = pd.concat([pd.read_csv(f) for f in files],ignore_index=True)

df['time'] = pd.to_datetime(df['time'])
df = df.sort_values(by='time')

print(f"Loaded shape: {df.shape}")

Loaded shape: (1148400, 39)


### 1.2 Time-Based Split & Sample

In [15]:
# Time-based split: Sep–Jan train, Feb test
train = df[df['time'].dt.month != 2]

# Use 20% sample for faster tuning
train_sample = train.sample(frac=0.2, random_state=42)

# Define features (exclude targets, time, identifiers)
EXCLUDE = {'time','zone_id','occupancy','volume','duration','e_price','s_price'}

FEATURES = train_sample.select_dtypes(include=np.number).columns.difference(EXCLUDE)

X_train = train_sample[FEATURES].astype(np.float32)
y_train = train_sample['occupancy'].astype(np.float32)

print("Training sample:", X_train.shape)

Training sample: (192720, 32)


### 1.3 TimeSeriesSplit

In [16]:
tscv = TimeSeriesSplit(n_splits=3)
print(f"Cross-validation: TimeSeriesSplit with {tscv.n_splits} folds")

Cross-validation: TimeSeriesSplit with 3 folds


## 2. Grid Search — Random Forest

In [5]:
t0 = timer.time()

rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [15, 20],
    'min_samples_leaf': [5, 10],
}

rf_grid = GridSearchCV(
    estimator=RandomForestRegressor(n_jobs=-1, random_state=42),
    param_grid=rf_param_grid,
    cv=tscv,
    scoring='r2',
    verbose=1,
    n_jobs=-1,
)

rf_grid.fit(X_train, y_train)

print(f"\nBest RF Params: {rf_grid.best_params_}")
print(f"Best RF CV R²: {rf_grid.best_score_:.4f}")
print(f"Time: {timer.time() - t0:.1f}s")

Fitting 3 folds for each of 8 candidates, totalling 24 fits

Best RF Params: {'max_depth': 15, 'min_samples_leaf': 5, 'n_estimators': 200}
Best RF CV R²: 0.9973
Time: 367.4s


## 3. Grid Search — XGBoost

In [17]:
t0 = timer.time()
    
xgb_param_grid = {
    'n_estimators': [300, 500, 800],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [5, 6, 8],
}
    
xgb_grid = GridSearchCV(
    estimator=XGBRegressor(
        tree_method='hist',
        n_jobs=-1,
        random_state=42,
        verbosity=0,
    ),
    param_grid=xgb_param_grid,
    cv=tscv,
    scoring='r2',
    verbose=1,
    n_jobs=-1,
)
    
xgb_grid.fit(X_train, y_train)
    
print(f"\nBest XGB Params: {xgb_grid.best_params_}")
print(f"Best XGB CV R²: {xgb_grid.best_score_:.4f}")
print(f"Time: {timer.time() - t0:.1f}s")

Fitting 3 folds for each of 27 candidates, totalling 81 fits

Best XGB Params: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 800}
Best XGB CV R²: 0.9988
Time: 62.9s


## 4. Grid Search — LightGBM

In [18]:
t0 = timer.time()
    
lgb_param_grid = {
    'n_estimators': [300, 500, 800],
    'learning_rate': [0.05, 0.1],
    'num_leaves': [31, 64],
}
    
lgb_grid = GridSearchCV(
    estimator=LGBMRegressor(
        n_jobs=-1,
        random_state=42,
        verbose=-1,
    ),
    param_grid=lgb_param_grid,
    cv=tscv,
    scoring='r2',
    verbose=1,
    n_jobs=-1,
)
    
lgb_grid.fit(X_train, y_train)
    
print(f"\nBest LGB Params: {lgb_grid.best_params_}")
print(f"Best LGB CV R²: {lgb_grid.best_score_:.4f}")
print(f"Time: {timer.time() - t0:.1f}s")

Fitting 3 folds for each of 12 candidates, totalling 36 fits

Best LGB Params: {'learning_rate': 0.1, 'n_estimators': 800, 'num_leaves': 31}
Best LGB CV R²: 0.9986
Time: 91.7s


## 5. Save Results

In [20]:
os.makedirs("../results", exist_ok=True)

# Build best params dictionary
best_params = {
    "RandomForest": rf_grid.best_params_,
    "XGBoost": xgb_grid.best_params_,
    "LightGBM": lgb_grid.best_params_
}

# Save as JSON (easy to load in Notebook 05)
with open("../results/best_hyperparameters.json", "w") as f:
    json.dump(best_params, f, indent=2)
print("Best hyperparameters saved to results/best_hyperparameters.json")

# Also save summary CSV
rows = [
    {
        "Model": name,
        "Best_Params": str(grid.best_params_),
        "Best_CV_R2": grid.best_score_}
    for name, grid in [
        ("RandomForest", rf_grid),
        ("XGBoost", xgb_grid),
        ("LightGBM", lgb_grid),
    ]
]

results_df = pd.DataFrame(rows)
results_df.to_csv("../results/grid_search_results.csv", index=False)
print("Grid search results saved to results/grid_search_results.csv")

Best hyperparameters saved to results/best_hyperparameters.json
Grid search results saved to results/grid_search_results.csv


In [24]:
print("Tuning Summary:\n")
for model, params in best_params.items():
    print(f"{model}:")
    for k, v in params.items():
        print(f"    {k}: {v}")
    print()

Tuning Summary:

RandomForest:
    max_depth: 15
    min_samples_leaf: 5
    n_estimators: 200

XGBoost:
    learning_rate: 0.05
    max_depth: 5
    n_estimators: 800

LightGBM:
    learning_rate: 0.1
    n_estimators: 800
    num_leaves: 31



## Notes

- Tuning was run on 20% of training data for speed — final models in Notebook 05 will train on 100% with these best parameters
- `device='cuda'`/`device='gpu'` removed to ensure portability across CPU/GPU machines
- TimeSeriesSplit respects temporal ordering (no future leakage in CV folds)